# RVC v2 Training Pipeline — Google Colab (L4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zakaria-kabir/Retrieval-based-Voice-Conversion-WebUI/blob/main/RVC_Colab_Pipeline.ipynb)

**Requirements before running:**

- Runtime -> Change runtime type -> **L4 GPU**
- Dataset `.tar.xz` files uploaded to Google Drive
- This repo pushed to GitHub with latest changes


## 1 - Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 2 - System Setup & Helpers


In [ ]:
import os, subprocess, sys

def run_live(cmd, cwd=None):
    """Run a shell command and stream stdout+stderr to the notebook in real time."""
    proc = subprocess.Popen(
        cmd, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, cwd=cwd
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc  # proc.returncode available after return

run_live("apt-get -y install -q build-essential python3-dev ffmpeg aria2")
run_live("nvidia-smi")
print(f"CPU cores: {os.cpu_count()}")


## 3 - Clone Repo


In [ ]:
GITHUB_REPO = "zakaria-kabir/Retrieval-based-Voice-Conversion-WebUI"
RVC_REPO    = "/content/Retrieval-based-Voice-Conversion-WebUI"

if not os.path.exists(RVC_REPO):
    run_live(f"git clone https://github.com/{GITHUB_REPO}.git {RVC_REPO}")
else:
    print("Repo already cloned - pulling latest...")
    run_live("git pull", cwd=RVC_REPO)

os.chdir(RVC_REPO)
print(f"Working directory: {os.getcwd()}")


## 4 - Install Python Dependencies

> ~3-5 minutes. Only needed once per Colab session.


In [ ]:
run_live("pip uninstall -y jax jaxlib tensorboard tensorflow")
run_live(f"pip install -r {RVC_REPO}/requirements-py311_updated.txt")
run_live("pip install pyyaml tqdm matplotlib")
print("Dependencies installed")


## 5 - Download Pretrained Models

> Skips files that already exist. Change `SAMPLE_RATE` here if not using 48k.


In [ ]:
SAMPLE_RATE = "48k"  # @param ["32k", "40k", "48k"]

os.makedirs(f"{RVC_REPO}/assets/pretrained_v2", exist_ok=True)
os.makedirs(f"{RVC_REPO}/assets/hubert",        exist_ok=True)
os.makedirs(f"{RVC_REPO}/assets/rmvpe",         exist_ok=True)

HF_BASE = "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main"

def dl(url, dest_dir, out_name):
    dest = f"{dest_dir}/{out_name}"
    if not os.path.exists(dest):
        run_live(
            f'aria2c --console-log-level=warn -c -x 16 -s 16 -k 1M'
            f' "{url}" -d "{dest_dir}" -o "{out_name}"'
        )
        print(f"Downloaded {out_name}")
    else:
        print(f"Skipping {out_name} (already exists)")

dl(f"{HF_BASE}/pretrained_v2/f0G{SAMPLE_RATE}.pth", f"{RVC_REPO}/assets/pretrained_v2", f"f0G{SAMPLE_RATE}.pth")
dl(f"{HF_BASE}/pretrained_v2/f0D{SAMPLE_RATE}.pth", f"{RVC_REPO}/assets/pretrained_v2", f"f0D{SAMPLE_RATE}.pth")
dl(f"{HF_BASE}/hubert_base.pt",                      f"{RVC_REPO}/assets/hubert",        "hubert_base.pt")
dl(f"{HF_BASE}/rmvpe.pt",                            f"{RVC_REPO}/assets/rmvpe",         "rmvpe.pt")

print("Pretrained models ready")


## 6 - Configure Speaker

> **Edit the top 3 variables for every new speaker you train.**


In [ ]:
# =============================================
#  EDIT THESE FOR EACH SPEAKER
SPEAKER_ID   = "sp021"   # @param {type:"string"}
DRIVE_TAR    = f"/content/drive/MyDrive/RVC_Datasets/{SPEAKER_ID}.tar.xz"  # @param {type:"string"}
DRIVE_BACKUP = "/content/drive/MyDrive/RVC_Backups"  # @param {type:"string"}
# =============================================

DATASET_DIR = "/content/dataset"
CONFIG_PATH = f"/content/{SPEAKER_ID}_config.yaml"

import yaml

config = {
    "rvc_repo_root":       RVC_REPO,
    "backup_root":         DRIVE_BACKUP,
    "dataset_tar":         f"{DATASET_DIR}/{SPEAKER_ID}.tar.xz",
    "dataset_extract_dir": DATASET_DIR,
    "model_name":          SPEAKER_ID,
    "speaker_id":          0,
    "sample_rate":         SAMPLE_RATE,
    "f0_method":           "rmvpe_gpu",
    "gpu":                 "0",
    "batch_size":          32,
    "total_epoch":         300,
    "save_every":          20,
    "version":             "v2",
    "use_f0":              True,
    "cache_data_in_gpu":   True,
    "save_latest_only":    False,
    "save_every_weights":  True,
    "preprocess_per":      3.7,
    "no_parallel":         False,
    "threads":             os.cpu_count() or 4,
    "inference": {
        "model_weight": "", "index_file": "",
        "input_audio": "", "output_dir": "/content/inference_output",
        "output_format": "wav", "f0_up_key": 0, "f0_method": "rmvpe",
        "index_rate": 0.75, "filter_radius": 3, "resample_sr": 0,
        "rms_mix_rate": 0.25, "protect": 0.33, "f0_file": "",
        "randomize_params": False, "random_ranges": {},
    }
}

with open(CONFIG_PATH, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Config written to {CONFIG_PATH}")
print(f"  Speaker:    {SPEAKER_ID}")
print(f"  Drive tar:  {DRIVE_TAR}")
print(f"  Backup dir: {DRIVE_BACKUP}/{SPEAKER_ID}")


## 7 - Copy Dataset from Drive to Colab SSD

> Training directly from Drive is ~10x slower -- always copy to `/content/` first.


In [ ]:
os.makedirs(DATASET_DIR, exist_ok=True)
local_tar = f"{DATASET_DIR}/{SPEAKER_ID}.tar.xz"

if not os.path.exists(local_tar):
    print(f"Copying {DRIVE_TAR} -> {local_tar}  (may take a few minutes)...")
    run_live(f'rsync -ah --info=progress2 "{DRIVE_TAR}" "{local_tar}"')
    print("Copy complete")
else:
    print(f"Already on local SSD: {local_tar}")


## 8 - Preflight Check


In [ ]:
run_live(
    f"python {RVC_REPO}/rvc_pipeline.py --config {CONFIG_PATH} train --dry-run",
    cwd=RVC_REPO
)


## 9 - Run Full Training Pipeline

> Runs all stages automatically:
> **Extract -> Preprocess -> F0 -> Features -> Filelist -> Train -> FAISS Index -> Analyze -> Backup -> Cleanup**

A **background thread** backs up to Drive **every 30 minutes** while training runs,
so a disconnect never loses more than ~30 minutes of progress.

**What gets saved to `RVC_Backups/{SPEAKER_ID}/` on Drive:**

| File | When saved |
|------|------------|
| `sp021_e20.pth`, `sp021_e40.pth`, ... | Every `save_every` epochs (small inference weights) |
| `G_200.pth`, `D_200.pth` | Latest training checkpoints (for resume) |
| `added_IVF*.index` | After training finishes (FAISS retrieval index) |
| `config.json`, `filelist.txt` | After training finishes |
| `train_stdout.log`, `training_metrics.csv` | After training finishes |


In [ ]:
import threading
from datetime import datetime

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BACKUP_INTERVAL_SEC = 30 * 60  # backup every 30 minutes

stop_backup = threading.Event()

def _periodic_backup():
    """Runs backup to Drive every BACKUP_INTERVAL_SEC while training is active."""
    while not stop_backup.wait(timeout=BACKUP_INTERVAL_SEC):
        ts = datetime.now().strftime("%H:%M:%S")
        print(f"[{ts}] Periodic backup starting...", flush=True)
        proc = run_live(
            f"python {RVC_REPO}/rvc_pipeline.py --config {CONFIG_PATH} backup",
            cwd=RVC_REPO
        )
        if proc.returncode == 0:
            print(f"[{ts}] Backup to Drive complete.", flush=True)
        else:
            print(f"[{ts}] Backup failed (will retry in {BACKUP_INTERVAL_SEC // 60} min).", flush=True)

backup_thread = threading.Thread(target=_periodic_backup, daemon=True)
backup_thread.start()
print(f"Periodic Drive backup started (every {BACKUP_INTERVAL_SEC // 60} min)")

try:
    result = run_live(
        f"python {RVC_REPO}/rvc_pipeline.py --config {CONFIG_PATH} train",
        cwd=RVC_REPO
    )
finally:
    stop_backup.set()
    backup_thread.join(timeout=10)

if result.returncode != 0:
    print("Pipeline failed. Check output above.")
else:
    print("Training pipeline complete!")


## 10 - (Optional) Plot Training Metrics


In [ ]:
run_live(
    f"python {RVC_REPO}/rvc_pipeline.py --config {CONFIG_PATH} analyze",
    cwd=RVC_REPO
)


## 11 - (Optional) Manual Backup

> The pipeline auto-backs up at the end of training, and the periodic thread
> saves every 30 min during training. Use this cell to trigger a one-shot backup manually.


In [ ]:
run_live(
    f"python {RVC_REPO}/rvc_pipeline.py --config {CONFIG_PATH} backup",
    cwd=RVC_REPO
)


---

## Resume After Disconnect

Because the periodic backup saves to Drive every 30 minutes, reconnecting loses at most
~30 min of training. The RVC train script auto-detects the latest G_/D_ checkpoint and
resumes from there.

**Steps:**
1. Rerun cells 1-6 (setup + config - sets `RVC_REPO`, `CONFIG_PATH`, etc.)
2. Rerun cell 7 only if `/content/dataset/` was wiped (check: `ls /content/dataset/`)
3. Run the **Restore** cell below
4. Run the **Resume Training** cell

> The FAISS index is built only after training finishes. If you disconnected before that,
> re-run cell 9 with the same config -- it will resume from the checkpoint, finish training,
> then build the index and do a final backup.


In [ ]:
# Restore latest checkpoints + weights from Drive back into the repo
run_live(
    f"python {RVC_REPO}/rvc_pipeline.py --config {CONFIG_PATH} restore",
    cwd=RVC_REPO
)


In [ ]:
# Resume training only (skips preprocess/f0/features -- already done before disconnect)
import threading
from datetime import datetime

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

TOTAL_EPOCH = config["total_epoch"]
SAVE_EVERY  = config["save_every"]
BATCH_SIZE  = config["batch_size"]
BACKUP_INTERVAL_SEC = 30 * 60

stop_backup = threading.Event()

def _periodic_backup():
    while not stop_backup.wait(timeout=BACKUP_INTERVAL_SEC):
        ts = datetime.now().strftime("%H:%M:%S")
        print(f"[{ts}] Periodic backup...", flush=True)
        run_live(
            f"python {RVC_REPO}/rvc_pipeline.py --config {CONFIG_PATH} backup",
            cwd=RVC_REPO
        )

backup_thread = threading.Thread(target=_periodic_backup, daemon=True)
backup_thread.start()

try:
    run_live(
        f"python infer/modules/train/train.py "
        f"-e {SPEAKER_ID} -sr {SAMPLE_RATE} -f0 1 "
        f"-bs {BATCH_SIZE} -g 0 -te {TOTAL_EPOCH} -se {SAVE_EVERY} "
        f"-pg assets/pretrained_v2/f0G{SAMPLE_RATE}.pth "
        f"-pd assets/pretrained_v2/f0D{SAMPLE_RATE}.pth "
        f"-l 0 -c 1 -sw 1 -v v2",
        cwd=RVC_REPO
    )
finally:
    stop_backup.set()
    backup_thread.join(timeout=10)

# After training finishes, build index + final backup
run_live(f"python train_index.py -e {SPEAKER_ID} -v v2", cwd=RVC_REPO)
run_live(
    f"python {RVC_REPO}/rvc_pipeline.py --config {CONFIG_PATH} backup",
    cwd=RVC_REPO
)


---

## Batch: Train Multiple Speakers

> Trains speakers sequentially, cleaning up Colab SSD between each one.
> Configure `SPEAKERS` and `DRIVE_DATASET_ROOT` before running.


In [ ]:
import shutil, glob

SPEAKERS           = ["sp021", "sp022", "sp023"]  # edit as needed
DRIVE_DATASET_ROOT = "/content/drive/MyDrive/RVC_Datasets"
DRIVE_BACKUP       = "/content/drive/MyDrive/RVC_Backups"

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

for spk in SPEAKERS:
    print(f"\n{'=' * 60}\nTraining: {spk}\n{'=' * 60}")

    cfg = config.copy()
    cfg["model_name"]  = spk
    cfg["dataset_tar"] = f"{DATASET_DIR}/{spk}.tar.xz"
    cfg["backup_root"] = DRIVE_BACKUP
    cfg_path = f"/content/{spk}_config.yaml"
    with open(cfg_path, "w") as f:
        yaml.dump(cfg, f)

    local_tar = f"{DATASET_DIR}/{spk}.tar.xz"
    if not os.path.exists(local_tar):
        run_live(f'rsync -ah --info=progress2 "{DRIVE_DATASET_ROOT}/{spk}.tar.xz" "{local_tar}"')

    proc = run_live(
        f"python {RVC_REPO}/rvc_pipeline.py --config {cfg_path} train",
        cwd=RVC_REPO
    )

    if proc.returncode != 0:
        print(f"FAILED: {spk} -- continuing to next speaker")
    else:
        shutil.rmtree(f"{DATASET_DIR}/{spk}", ignore_errors=True)
        if os.path.exists(local_tar):
            os.remove(local_tar)
        for d in glob.glob(f"{RVC_REPO}/logs/{spk}/0_gt_wavs"):
            shutil.rmtree(d, ignore_errors=True)
        print(f"Done: {spk} -- cleaned up SSD")
